<a href="https://colab.research.google.com/github/fernandodeeke/sistemas_dinamicos/blob/main/dois_tanques_buttons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import ipywidgets as widgets
from ipywidgets import interact


def simular_sistema(
    a=0.10,
    b=0.075,
    c=1.50,
    d=0.10,
    e=0.20,
    f=3.00,
    q1_0=25.0,
    q2_0=15.0,
    tf=80.0,
    n_pontos=1200
):
    """
    Sistema linear não homogêneo:

        dq1/dt = -a q1 + b q2 + c
        dq2/dt =  d q1 - e q2 + f

    Para os valores originais:

        a = 1/10 = 0.10
        b = 1.5/20 = 0.075
        c = 1.5
        d = 1/10 = 0.10
        e = 1/5 = 0.20
        f = 3.0
    """

    def sistema(t, q):
        q1, q2 = q
        dq1 = -a*q1 + b*q2 + c
        dq2 =  d*q1 - e*q2 + f
        return [dq1, dq2]

    # Condições iniciais
    q0 = [q1_0, q2_0]

    # Malha temporal
    t0 = 0.0
    t_eval = np.linspace(t0, tf, int(n_pontos))

    # Integração numérica
    sol = solve_ivp(
        sistema,
        (t0, tf),
        q0,
        t_eval=t_eval,
        method="RK45"
    )

    t = sol.t
    q1 = sol.y[0]
    q2 = sol.y[1]

    # Matriz do sistema linear: q' = A q + B
    A = np.array([
        [-a,  b],
        [ d, -e]
    ])

    B = np.array([c, f])

    # Ponto de equilíbrio: A q* + B = 0
    # Portanto: A q* = -B
    try:
        q_eq = np.linalg.solve(A, -B)
        q1_eq, q2_eq = q_eq
        existe_equilibrio = True
    except np.linalg.LinAlgError:
        q1_eq, q2_eq = np.nan, np.nan
        existe_equilibrio = False

    # Gráfico das séries temporais
    plt.figure(figsize=(9, 5.2), dpi=180)

    plt.plot(t, q1, label=r"$q_1(t)$")
    plt.plot(t, q2, label=r"$q_2(t)$")

    if existe_equilibrio:
        plt.axhline(
            q1_eq,
            linestyle="--",
            linewidth=1.2,
            label=rf"$q_1^*={q1_eq:.2f}$"
        )

        plt.axhline(
            q2_eq,
            linestyle=":",
            linewidth=1.5,
            label=rf"$q_2^*={q2_eq:.2f}$"
        )

    plt.xlabel(r"$t$")
    plt.ylabel("Valor")
    plt.title(
        rf"Séries temporais para $q_1(0)={q1_0:.1f}$, "
        rf"$q_2(0)={q2_0:.1f}$"
    )

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    if existe_equilibrio:
        print("Ponto de equilíbrio:")
        print(f"q1* = {q1_eq:.6f}")
        print(f"q2* = {q2_eq:.6f}")
    else:
        print("A matriz do sistema é singular. O ponto de equilíbrio não é único ou pode não existir.")


interact(
    simular_sistema,

    a=widgets.FloatSlider(
        value=0.10, min=0.00, max=0.50, step=0.01,
        description="a",
        continuous_update=False
    ),

    b=widgets.FloatSlider(
        value=0.075, min=-0.50, max=0.50, step=0.005,
        description="b",
        continuous_update=False
    ),

    c=widgets.FloatSlider(
        value=1.50, min=-10.00, max=10.00, step=0.10,
        description="c",
        continuous_update=False
    ),

    d=widgets.FloatSlider(
        value=0.10, min=-0.50, max=0.50, step=0.01,
        description="d",
        continuous_update=False
    ),

    e=widgets.FloatSlider(
        value=0.20, min=0.00, max=0.80, step=0.01,
        description="e",
        continuous_update=False
    ),

    f=widgets.FloatSlider(
        value=3.00, min=-10.00, max=10.00, step=0.10,
        description="f",
        continuous_update=False
    ),

    q1_0=widgets.FloatSlider(
        value=25.0, min=-50.0, max=100.0, step=1.0,
        description="q1(0)",
        continuous_update=False
    ),

    q2_0=widgets.FloatSlider(
        value=15.0, min=-50.0, max=100.0, step=1.0,
        description="q2(0)",
        continuous_update=False
    ),

    tf=widgets.FloatSlider(
        value=80.0, min=5.0, max=200.0, step=5.0,
        description="tf",
        continuous_update=False
    ),

    n_pontos=widgets.IntSlider(
        value=1200, min=200, max=3000, step=100,
        description="pontos",
        continuous_update=False
    )
);

interactive(children=(FloatSlider(value=0.1, continuous_update=False, description='a', max=0.5, step=0.01), Fl…

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import ipywidgets as widgets
from ipywidgets import interact


def retrato_de_fase_interativo(
    a=0.10,
    b=0.075,
    c=1.50,
    d=0.10,
    e=0.20,
    f=3.00,
    q1_0=25.0,
    q2_0=15.0,
    tf=80.0,
    n_pontos=1200,
    densidade=22,
    margem=8.0
):
    """
    Sistema:
        dq1/dt = -a q1 + b q2 + c
        dq2/dt =  d q1 - e q2 + f
    """

    def sistema(t, q):
        q1, q2 = q
        dq1 = -a*q1 + b*q2 + c
        dq2 =  d*q1 - e*q2 + f
        return [dq1, dq2]

    # Condição inicial
    q0 = [q1_0, q2_0]

    # Intervalo de tempo
    t0 = 0.0
    t_eval = np.linspace(t0, tf, int(n_pontos))

    # Integração numérica
    sol = solve_ivp(
        sistema,
        (t0, tf),
        q0,
        t_eval=t_eval,
        method="RK45"
    )

    t = sol.t
    q1 = sol.y[0]
    q2 = sol.y[1]

    # Matriz do sistema linear: q' = A q + B
    A = np.array([
        [-a,  b],
        [ d, -e]
    ])
    B = np.array([c, f])

    # Ponto de equilíbrio: A q* + B = 0
    try:
        q_eq = np.linalg.solve(A, -B)
        q1_eq, q2_eq = q_eq
        existe_equilibrio = True
    except np.linalg.LinAlgError:
        q1_eq, q2_eq = np.nan, np.nan
        existe_equilibrio = False

    # Limites do gráfico
    if existe_equilibrio:
        q1_min = min(q1.min(), q1_eq, q0[0]) - margem
        q1_max = max(q1.max(), q1_eq, q0[0]) + margem
        q2_min = min(q2.min(), q2_eq, q0[1]) - margem
        q2_max = max(q2.max(), q2_eq, q0[1]) + margem
    else:
        q1_min = min(q1.min(), q0[0]) - margem
        q1_max = max(q1.max(), q0[0]) + margem
        q2_min = min(q2.min(), q0[1]) - margem
        q2_max = max(q2.max(), q0[1]) + margem

    # Malha do campo vetorial
    Q1, Q2 = np.meshgrid(
        np.linspace(q1_min, q1_max, int(densidade)),
        np.linspace(q2_min, q2_max, int(densidade))
    )

    dQ1 = -a*Q1 + b*Q2 + c
    dQ2 =  d*Q1 - e*Q2 + f

    # Normalização para mostrar apenas a direção do campo
    norm = np.sqrt(dQ1**2 + dQ2**2)
    norm[norm == 0] = 1.0

    # Gráfico
    plt.figure(figsize=(6.8, 6.2), dpi=180)

    plt.quiver(
        Q1, Q2,
        dQ1 / norm, dQ2 / norm,
        angles="xy"
    )

    plt.plot(q1, q2, linewidth=2, label="Trajetória")
    plt.plot(
        q0[0], q0[1],
        marker="o", linestyle="None", markersize=7,
        label="Condição inicial"
    )

    if existe_equilibrio:
        plt.plot(
            q1_eq, q2_eq,
            marker="x", linestyle="None",
            markersize=10, mew=2,
            label="Equilíbrio"
        )

    plt.xlabel(r"$q_1$")
    plt.ylabel(r"$q_2$")
    plt.title(
        rf"Retrato de fase para $q_1(0)={q1_0:.1f},\; q_2(0)={q2_0:.1f}$"
    )
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    if existe_equilibrio:
        print("Ponto de equilíbrio:")
        print(f"q1* = {q1_eq:.6f}")
        print(f"q2* = {q2_eq:.6f}")
    else:
        print("A matriz do sistema é singular. O ponto de equilíbrio não é único ou pode não existir.")


interact(
    retrato_de_fase_interativo,

    a=widgets.FloatSlider(
        value=0.10, min=0.00, max=0.50, step=0.01,
        description="a",
        continuous_update=False
    ),

    b=widgets.FloatSlider(
        value=0.075, min=-0.50, max=0.50, step=0.005,
        description="b",
        continuous_update=False
    ),

    c=widgets.FloatSlider(
        value=1.50, min=-10.00, max=10.00, step=0.10,
        description="c",
        continuous_update=False
    ),

    d=widgets.FloatSlider(
        value=0.10, min=-0.50, max=0.50, step=0.01,
        description="d",
        continuous_update=False
    ),

    e=widgets.FloatSlider(
        value=0.20, min=0.00, max=0.80, step=0.01,
        description="e",
        continuous_update=False
    ),

    f=widgets.FloatSlider(
        value=3.00, min=-10.00, max=10.00, step=0.10,
        description="f",
        continuous_update=False
    ),

    q1_0=widgets.FloatSlider(
        value=25.0, min=-50.0, max=100.0, step=1.0,
        description="q1(0)",
        continuous_update=False
    ),

    q2_0=widgets.FloatSlider(
        value=15.0, min=-50.0, max=100.0, step=1.0,
        description="q2(0)",
        continuous_update=False
    ),

    tf=widgets.FloatSlider(
        value=80.0, min=5.0, max=200.0, step=5.0,
        description="tf",
        continuous_update=False
    ),

    n_pontos=widgets.IntSlider(
        value=1200, min=200, max=3000, step=100,
        description="trajetória",
        continuous_update=False
    ),

    densidade=widgets.IntSlider(
        value=22, min=8, max=40, step=2,
        description="malha",
        continuous_update=False
    ),

    margem=widgets.FloatSlider(
        value=8.0, min=2.0, max=20.0, step=1.0,
        description="margem",
        continuous_update=False
    )
);

interactive(children=(FloatSlider(value=0.1, continuous_update=False, description='a', max=0.5, step=0.01), Fl…